---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [118]:
# importar librerías
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

In [119]:
# cargar archivos
orders = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv")
catalog = pd.read_csv ("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv")
marketing = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv")

In [120]:
print("Pedidos:", orders.shape)
print("Catálogo:", catalog.shape)
print("Marketing:", marketing.shape)

Pedidos: (25100, 12)
Catálogo: (7, 4)
Marketing: (1620, 5)


In [121]:
# explorar datasets
orders.info()
orders.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [122]:
catalog.info()
catalog.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 356.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [123]:
marketing.info()
marketing.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas
---

In [124]:
orders['fecha_hora_pedido']=pd.to_datetime(orders['fecha_hora_pedido'])
orders.info()
orders_copy = orders.copy()
orders_copy.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB


,fecha_hora_pedido,cantidad,precio_unitario,monto_descuento,monto_total
count,25100,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,2025-04-01 03:21:22.231075584,7.092735,259.305549,4.500798,2.072680e+03
min,2025-01-01 00:00:00,-2.000000,20.030000,0.000000,-4.926500e+02
25%,2025-02-15 00:00:00,1.000000,138.377500,0.000000,1.805075e+02
50%,2025-04-02 00:00:00,2.000000,258.715000,0.000000,3.417500e+02
75%,2025-05-16 00:00:00,2.000000,380.332500,10.000000,5.185800e+02
max,2025-06-30 00:00:00,20000.000000,499.960000,15.000000,8.840200e+06
std,NaN,296.277003,138.726461,5.223010,9.894995e+04


En la celda de arriba se realizó un cambio de tipo de dato en la columna fecha, se cambio de objetc a datetime. Adicional, se comprobo que el cambio quedara con orders.info(). Asi mismo, se realizó una copia del DF orders y se utilizó la funcion .describe() para revisar si las columnas numericas tenian valores negativos o ceros invalidos, con este ultimo se evidencia que si se presentan valor negativos en cantidad ya que el minimo llega hasta -2  y tambien en monto_total ya que el minimo en este es -4.92... Por otro lado en monto_descuento se evidencian ceros pero en esta columna si es mas probable que estos valores se presenten ya que puede no aplicarse ningun descuento.

En el codigo de abajo de esta celda de texto, se evidencia la funcion para sacar la sumatoria de nulos por columnas de la tabla orders

In [125]:
orders_copy.isna().sum()

,0
id_pedido,0
id_usuario,0
fecha_hora_pedido,0
pais,300
dispositivo,20
fuente_referencia,30
nombre_producto,30
categoria_producto,80
cantidad,50
precio_unitario,50


In [126]:
catalog_copy = catalog.copy()
catalog_copy.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


En la celda de arriba, se aplica unicamente el metodo describe para comprobas valores negativos y ceros, en este no se aplica la funcion de nulos ya que esta tabla no posee nulos.

In [127]:
marketing['fecha']=pd.to_datetime(marketing['fecha'])
marketing.info()
marketing_copy = marketing.copy()
marketing_copy.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


,fecha,gasto
count,1620,1620.00000
mean,2025-03-31 12:00:00,1772.74292
min,2025-01-01 00:00:00,501.11000
25%,2025-02-14 18:00:00,1128.03000
50%,2025-03-31 12:00:00,1782.42500
75%,2025-05-15 06:00:00,2420.68500
max,2025-06-29 00:00:00,2999.36000
std,NaN,734.43294


En la celda de arriba se observa que ya el dtype de fecha fue modificado y que la funcion describe no presenta valores negativos o ceros

In [128]:
orders_copy.duplicated().sum()
orders_copy[orders_copy.duplicated(keep=False)]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
734,order_734,user_6347,2025-05-02,Colombia,desktop,organic,Blender-XL-Red,Hogar,1.0,58.00,0.0,58.00
812,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32
974,order_974,user_3262,2025-05-04,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,2.0,268.58,5.0,532.16
1361,order_1361,user_507,2025-06-27,Mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,2.0,239.61,0.0,479.22
1735,order_1735,user_728,2025-05-13,Mexico,mobile,social,Tablet-Standard-64GB,Electronica,2.0,419.04,0.0,838.08
...,...,...,...,...,...,...,...,...,...,...,...,...
25095,order_3913,user_380,2025-02-18,Argentina,desktop,paid_search,Phone-Pro-128GB,Electronica,1.0,82.28,0.0,82.28
25096,order_23405,user_7833,2025-04-04,Colombia,mobile,paid_search,Phone-Pro-128GB,Electronica,2.0,99.25,5.0,193.50
25097,order_5615,user_5417,2025-05-13,Colombia,desktop,social,Blender-XL-Red,Hogar,2.0,450.35,5.0,895.69
25098,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32


In [129]:
orders_copy = orders_copy.drop_duplicates()
orders_copy.duplicated().sum()

np.int64(0)

In [130]:
print("Catalogo:", catalog_copy.duplicated().sum())
print("Marketing:", marketing_copy.duplicated().sum())

Catalogo: 0
Marketing: 0


La siguiente función permite revisar las variables categoricas, en esta se usa .value_counts para contar apariciones de cada etiqueta y se usa dropna=Flase par calcular valores faltantes.

In [131]:
columnas = [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto'
]

for columna in columnas:
    print(f"\nColumna: {columna}")
    print(orders_copy[columna].value_counts(dropna=False))



Columna: pais
pais
Colombia     7481
Mexico       7478
Argentina    7259
mexico        863
colombia      823
argentina     796
NaN           300
Name: count, dtype: int64

Columna: dispositivo
dispositivo
desktop    12711
mobile     12269
NaN           20
Name: count, dtype: int64

Columna: fuente_referencia
fuente_referencia
social         8402
organic        8288
paid_search    8280
NaN              30
Name: count, dtype: int64

Columna: nombre_producto
nombre_producto
Blender-XL-Red          4179
Jacket-Winter-M         4176
Vacuum-Pro-Black        4176
Sneakers-Urban-42       4148
Laptop-Gaming-16GB      2782
Tablet-Standard-64GB    2769
Phone-Pro-128GB         2740
NaN                       30
Name: count, dtype: int64

Columna: categoria_producto
categoria_producto
Hogar          8346
Moda           8295
Electronica    8279
NaN              80
Name: count, dtype: int64


Aquí encontramos una inconsistencia en pais: cada país aparece con mayúscula y con minúscula. Por ejemplo, Colombia y colombia deben pertenecer al mismo grupo.
Para unificarlos así:
- .str.strip() elimina espacios al principio y al final, si existen.
- .str.capitalize() deja la primera letra en mayúscula y las demás en minúscula.

Con la función de abajo se unifica la escritura para la columna 'pais' en donde con .str.strip() se eliminan los espacios al principio y al final y con .str.capitalize se deja unicamente la primera letra en mayuscula, adicional se comprueba que el resultado funcionará. Se evidencia que ya queda unificado como solo Colombia

In [132]:
orders_copy['pais'] = orders_copy['pais'].str.strip().str.capitalize()
orders_copy['pais'].value_counts(dropna=False)

,count
pais,
Mexico,8341
Colombia,8304
Argentina,8055
NaN,300


In [133]:
orders_copy['categoria_producto'] = (
    orders_copy['categoria_producto']
    .replace({'Electronica': 'Electrónica'}))

In [134]:
catalog_copy['categoria_producto'].value_counts(dropna=False)

,count
categoria_producto,
Electrónica,3
Hogar,2
Moda,2


Se evidenciaba diferencia en la etiqueta de Electrónica y Electronica sin tilde entre las tablas de pedidos y catalogo, con la funcion de arriba se unifica la escritura de la palabra en las dos tablas

In [135]:
for columna in ['nombre_producto', 'categoria_producto', 'proveedor']:
    print(f"\nCatálogo — {columna}")
    print(catalog_copy[columna].value_counts(dropna=False))

for columna in ['pais', 'canal', 'id_campaña']:
    print(f"\nMarketing — {columna}")
    print(marketing_copy[columna].value_counts(dropna=False))


Catálogo — nombre_producto
nombre_producto
Laptop-Gaming-16GB      1
Phone-Pro-128GB         1
Tablet-Standard-64GB    1
Blender-XL-Red          1
Vacuum-Pro-Black        1
Sneakers-Urban-42       1
Jacket-Winter-M         1
Name: count, dtype: int64

Catálogo — categoria_producto
categoria_producto
Electrónica    3
Hogar          2
Moda           2
Name: count, dtype: int64

Catálogo — proveedor
proveedor
Fuller, Pena and Myers     1
King Ltd                   1
Bowers LLC                 1
Long-Reid                  1
Rivera, Carr and Finley    1
Greene-Smith               1
Mcmillan-Rhodes            1
Name: count, dtype: int64

Marketing — pais
pais
Mexico       540
Colombia     540
Argentina    540
Name: count, dtype: int64

Marketing — canal
canal
paid_search    507
organic        506
social         506
NaN            101
Name: count, dtype: int64

Marketing — id_campaña
id_campaña
organic_Mexico           180
paid_search_Mexico       180
social_Mexico            180
organic_Col

In [136]:
print(marketing_copy[columna].value_counts(dropna=False))
orders_copy.isna().sum()

id_campaña
organic_Mexico           180
paid_search_Mexico       180
social_Mexico            180
organic_Colombia         180
paid_search_Colombia     180
social_Colombia          180
organic_Argentina        180
paid_search_Argentina    180
social_Argentina         180
Name: count, dtype: int64


,0
id_pedido,0
id_usuario,0
fecha_hora_pedido,0
pais,300
dispositivo,20
fuente_referencia,30
nombre_producto,30
categoria_producto,80
cantidad,50
precio_unitario,50


In [137]:
columnas = ['cantidad', 'precio_unitario', 'monto_descuento']

orders_copy.loc[
    orders_copy[columnas].isna().any(axis=1),
    columnas + ['monto_total']
]

,cantidad,precio_unitario,monto_descuento,monto_total
74,NaN,NaN,NaN,595.85
75,NaN,NaN,NaN,458.15
76,NaN,NaN,NaN,319.75
77,NaN,NaN,NaN,227.55
78,NaN,NaN,NaN,432.39
79,NaN,NaN,NaN,527.15
80,NaN,NaN,NaN,711.18
81,NaN,NaN,NaN,41.08
82,NaN,NaN,NaN,431.55
83,NaN,NaN,NaN,279.35


# Evaluación de cantidades negativas

In [138]:

orders_copy.loc[orders_copy['cantidad']<0,
    ['id_pedido', 'cantidad','precio_unitario','monto_descuento', 'monto_total']]


,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,-2.0,101.31,10.0,-192.62
267,order_267,-1.0,43.50,5.0,-38.50
268,order_268,-1.0,497.65,5.0,-492.65
269,order_269,-1.0,423.53,0.0,-423.53


# Evaluación de cantidad máxima

In [139]:
orders_copy.loc[orders_copy['cantidad']==orders_copy['cantidad'].max(),
    ['id_pedido','cantidad','precio_unitario','monto_descuento','monto_total']]

,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total
3656,order_3656,20000.0,297.66,0.0,5953200.0
3668,order_3668,20000.0,348.31,0.0,6966200.0
3722,order_3722,20000.0,442.01,0.0,8840200.0
3726,order_3726,20000.0,290.85,0.0,5817000.0


In [140]:
orders_copy['cantidad'].value_counts(dropna=False).sort_index()

,count
cantidad,
-2.0,1
-1.0,3
1.0,12345
2.0,12591
10000.0,6
20000.0,4
NaN,50


Al observar los datos, se eviencia que exiten 4 datos con cantidad de 20000 los cualevaluando el contexto del negocio pensaria que son errores ya que para un negocio como Rappi una venta con cantidad de 20000 no hace mucho sentido por lo que los excluire y trabajare el data set con valores menores 10000, que es lo que se hace en la siguiente celda

In [141]:
orders_copy=orders_copy.loc[orders_copy['cantidad'].isna()
    |((orders_copy['cantidad']>0)&(orders_copy['cantidad']<10000))].copy()
orders_copy['cantidad'].value_counts(dropna=False).sort_index()


,count
cantidad,
1.0,12345
2.0,12591
NaN,50


En las Columnas 'pais', 'dispositivo'y 'fuente de referencia' quedan valores nulos, para evitar que estos valores se pierdan, se van a etiquetar como
`Desconocido` de tal manera que en el dashboard se puedan agrupar por el mismo nombre. El resto de valores nulos en categoria y valores numericos se quedara como nulos ya que presentan limitaciones y no se pueden sacar de la operacion con otras columnas o alguna otra alterativa

In [142]:
columnas = ['pais', 'dispositivo', 'fuente_referencia']

orders_copy[columnas] = orders_copy[columnas].fillna('Desconocido')
orders_copy.isna().sum()

,0
id_pedido,0
id_usuario,0
fecha_hora_pedido,0
pais,0
dispositivo,0
fuente_referencia,0
nombre_producto,30
categoria_producto,80
cantidad,50
precio_unitario,50


In [143]:
marketing_copy['canal'] = marketing_copy['canal'].fillna('Desconocido')

In [144]:
marketing_copy.groupby('canal')['gasto'].sum()

,gasto
canal,
Desconocido,177179.10
organic,913533.01
paid_search,863088.21
social,918043.21


Completé los valores vacíos de canal con la etiqueta “Desconocido” para incluirlos en la agrupación. Estos registros representan un gasto de 177.179,10. Al incluirlos, la suma del gasto por canal coincide con el gasto total de marketing: 2.871.843,53.

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [145]:
# exportar datasets
orders_copy.to_csv('orders_clean.csv', index=False)
catalog_copy.to_csv('catalog_clean.csv', index=False)
marketing_copy.to_csv('marketing_clean.csv', index=False)

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

# Parte 1: Rentabilidad del negocio

In [146]:
# ingreso total Revenue
revenue_total = orders_copy['monto_total'].sum()
revenue_total

np.float64(9643909.559999999)

In [147]:
#costo total
orders_cost = orders_copy.merge(catalog_copy[['nombre_producto','costo_unitario']], on='nombre_producto', how='left',validate='many_to_one')
orders_cost['costo_pedido']= orders_cost['cantidad']*orders_cost['costo_unitario']
costo_total= orders_cost['costo_pedido'].sum()
costo_total

np.float64(3828869.01)

En la celda anterior, se creo una nueva tabla `orders_cost` en la cual se unieron las tablas `orders_copy` que contiene la cantidad del pedido mientras que `catalog_copy` contiene el costo_unitario de cada producto. Para unirlo se uso .merge.
catalog_copy[[...]]: traer únicamente el nombre del producto y su costo.
on='nombre_producto': relacionar las filas que tienen el mismo producto.
how='left': conserva todos los pedidos, incluso los que no tienen producto; su costo quedará nulo.
validate='many_to_one': comprueba que cada producto aparece una sola vez en el catálogo, aunque tenga muchos pedidos

In [148]:
# ¿Cuánto se ha invertido en marketing? INVERSIÓN EN MARKETING
gasto_marketing = marketing_copy['gasto'].sum()
gasto_marketing

np.float64(2871843.53)

Como no todas las filas contienen datos de costos en `orders_cost` en la celda de abajo, se seleccionan unicamente los pedidos que si presentan costo para que las celdas vacias no vayan a interferir en los calculos `.notna` selecione filas cuyo costo pudo calcularse

In [149]:
#Se filtran unicamente las filas que contiene informacion del costo del pedido
pedidos_con_costo = orders_cost.loc[orders_cost['costo_pedido'].notna().copy()]

En la celda de abajo, se calcula los ingresos de los pedidos que si presenta información con respecto a costo

In [150]:
#Ingresos de pedido con costo conocido
revenue_con_costo = pedidos_con_costo['monto_total'].sum()
revenue_con_costo

np.float64(9610018.94)

In [151]:
# Resultado parcial: ingresos de pedidos con costo conocido menos costos y marketing

profit = revenue_con_costo - costo_total - gasto_marketing
profit


np.float64(2909306.4)

Para calcular el profit utilicé los ingresos de los pedidos cuyo costo pude determinar: 9.610.018,94. A ese valor le resté los costos calculados y el gasto total de marketing. El resultado fue 2.909.306,40.

Este resultado es parcial porque no incluye los ingresos de los pedidos sin costo conocido, aunque sí descuenta todo el gasto de marketing. Los ingresos totales de los pedidos limpios son 9.643.909,56 y se muestran por separado.

# Parte 2: comportamiento de ventas

In [152]:
#ticket promedio por orden
ticket_mean = orders_cost['monto_total'].mean()
ticket_mean

np.float64(385.97252701512843)

El promedio en ventas fue de 385.972

In [153]:
#Cantidad promedio de productos por orden
products_mean = orders_cost['cantidad'].mean()
products_mean

np.float64(1.5049326275264678)

La cantidad promedio de productos por orden fue de 1.504

In [154]:
#Cuál es el producto más vendido?

orders_cost.groupby('nombre_producto')['cantidad'].sum().idxmax()


'Vacuum-Pro-Black'

In [155]:
mayor_venta=orders_cost.groupby('nombre_producto')['cantidad'].sum()
ventas_ordenadas = mayor_venta.sort_values(ascending=False)
ventas_ordenadas.head(1)


,cantidad
nombre_producto,
Vacuum-Pro-Black,6284.0


El prodcuto con más ventas fue Vacuum-Pro-Black, en la primera celda se saca unicamente el nombre del producto con más ventas pero queria tener el datos de las ventas por lo que utilice la siguiente función para sacar nuevamente el nombre con su respectiva cantidad de ventas

In [156]:
#Cuanto se ha gastado en marketing por canal ?
marketing_copy.groupby('canal')['gasto'].sum()

,gasto
canal,
Desconocido,177179.10
organic,913533.01
paid_search,863088.21
social,918043.21


## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [157]:
%pip install "psycopg[binary]"

In [158]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [159]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [160]:
# Explorar tabla events
# =========================
query_events = '''
SELECT DISTINCT nombre_evento
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head(7)

,nombre_evento
0,add_payment_info
1,first_visit
2,begin_checkout
3,add_to_cart
4,select_item
5,purchase


In [161]:
# PARTE 1: Totales del funnel
# ======================



query_totals = '''
WITH etapa1 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'),
etapa2 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'),
etapa3 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'),
etapa4 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'),
etapa5 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE  nombre_evento = 'add_payment_info'),
etapa6 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE  nombre_evento = 'purchase')
SELECT
(SELECT COUNT (*) FROM etapa1) AS primera_visita,
(SELECT COUNT (*) FROM etapa2) AS seleccion_item,
(SELECT COUNT (*) FROM etapa3) AS anadir_carro,
(SELECT COUNT (*) FROM etapa4) AS checkout,
(SELECT COUNT (*) FROM etapa5) AS payment,
(SELECT COUNT (*) FROM etapa6) AS purchase;


'''


totals = pd.read_sql(query_totals, con=engine)
totals

,primera_visita,seleccion_item,anadir_carro,checkout,payment,purchase
0,7796,7582,7634,7208,6250,6240


La tabla de arriba muestra la cantidad de usuarios que se presentan en cada etapa del funnel, en esta se evidencia que en la etapa 3 hay un aumento en la cantidad pasando de 7582 a 7634. De acuerdo a esta información la hipotesis que manejo es que algunos de los usuarios que aparecen en la etapa de anadir_carro, no llegan directamente del paso anterior que es select_item, algunos de estos usuarios pueden acceder a esta etapa a través de un link o de una invitación diferente.

In [162]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH etapa1 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'),
etapa2 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
    AND id_usuario IN (SELECT id_usuario FROM etapa1)),
etapa3 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'
    AND id_usuario IN (SELECT id_usuario FROM etapa2)),
etapa4 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'
    AND id_usuario IN (SELECT id_usuario FROM etapa3)),
etapa5 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE  nombre_evento = 'add_payment_info'
    AND id_usuario IN (SELECT id_usuario FROM etapa4)),
etapa6 AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE  nombre_evento = 'purchase'
    AND id_usuario IN (SELECT id_usuario FROM etapa5))
SELECT
(SELECT COUNT (*) FROM etapa1) AS primera_visita,
(SELECT COUNT (*) FROM etapa2) AS seleccion_item,
(SELECT COUNT (*) FROM etapa2)*100.0/(SELECT COUNT (*) FROM etapa1) AS conversion_etapa_1,
(SELECT COUNT (*) FROM etapa3) AS anadir_carro,
(SELECT COUNT (*) FROM etapa3)*100.0/(SELECT COUNT (*) FROM etapa2) AS conversion_etapa_2,
(SELECT COUNT (*) FROM etapa4) AS checkout,
(SELECT COUNT (*) FROM etapa4)*100.0/(SELECT COUNT (*) FROM etapa3) AS conversion_etapa_3,
(SELECT COUNT (*) FROM etapa5) AS payment,
(SELECT COUNT (*) FROM etapa5)*100.0/(SELECT COUNT (*) FROM etapa4) AS conversion_etapa_4,
(SELECT COUNT (*) FROM etapa6) AS payment,
(SELECT COUNT (*) FROM etapa6)*100.0/(SELECT COUNT (*) FROM etapa5) AS conversion_etapa_5;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,primera_visita,seleccion_item,conversion_etapa_1,anadir_carro,conversion_etapa_2,checkout,conversion_etapa_3,payment,conversion_etapa_4,payment,conversion_etapa_5
0,7796,7393,94.830682,7052,95.387529,6364,90.243902,4967,78.048397,3857,77.652507


Se realizan las conversiones con respecto a cada etapa pero para evitar el problema que se estaba teniendo anteriormente de recibir en la etapa 3 usuarios provenientes de otros lados, se implementa la función en cada etapa que flitra los usuarios mientras estos cumplan con la condición de haber pasado por la etapa anterior.

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [163]:
# Explorar tabla users
# =========================
query_users = '''

SELECT *
FROM users;

'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [164]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;

'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [165]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte_mes
    FROM users
), actividad AS (
    SELECT
        id_usuario,
        FLOOR(dias_despues_registro / 7) AS semana_num,
        activo
    FROM user_activity
), base AS (
    SELECT
        c.cohorte_mes,
        c.id_usuario,
        ac.semana_num,
        ac.activo
    FROM cohortes c
    LEFT JOIN actividad ac
    ON c.id_usuario = ac.id_usuario
), retencion AS (
    SELECT
        cohorte_mes,
        COUNT(DISTINCT id_usuario) As clientes_iniciales,
        COUNT(
            DISTINCT CASE
                WHEN semana_num >= 1 AND activo = 1 THEN id_usuario END) AS retenidos_w1,
        COUNT(
            DISTINCT CASE
                WHEN semana_num >= 2 AND activo = 1 THEN id_usuario END) AS retenidos_w2,
        COUNT(
            DISTINCT CASE
                WHEN semana_num >= 3 AND activo = 1 THEN id_usuario END) AS retenidos_w3
    FROM base
    GROUP BY cohorte_mes
)
SELECT
    TO_CHAR(cohorte_mes, 'YYYY-MM') AS cohorte,
    clientes_iniciales,
    retenidos_w1,
    retenidos_w2,
    retenidos_w3,
    ROUND(retenidos_w1 * 1.0 / clientes_iniciales, 2) As semana_1,
    ROUND(retenidos_w2 * 1.0 / clientes_iniciales, 2) As semana_2,
    ROUND(retenidos_w3 * 1.0 / clientes_iniciales, 2) As semana_3
FROM retencion
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,retenidos_w1,retenidos_w2,retenidos_w3,semana_1,semana_2,semana_3
0,2025-01,1627,1381,1253,1027,0.85,0.77,0.63
1,2025-02,1444,1255,1154,940,0.87,0.80,0.65
2,2025-03,1636,1428,1306,1060,0.87,0.80,0.65
3,2025-04,1606,1394,1261,1022,0.87,0.79,0.64
4,2025-05,1687,1446,1321,1088,0.86,0.78,0.64


Se evidencia un comportamiento normal en los datos, es decir unos % más altos en la primera semana que van bajando con el paso de las semanas, adicional se puede establecer que llegando a la semana 3 estos datos se estabilizan, quedando todos alrededor del 0.64, 0.65

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La modificacion en la UI del checkout no impacto en la tasa de conversion de compra, es decir la tasa de conversion es igual para los dos casos.
   - **H₁ (Hipótesis alternativa):**  La tasa de conversión de compra es diferente entre el grupo que vio el checkout modificado y el grupo que vio la versión anterior.
   
**Test estadístico:** Z test ya que mide proporciones y se estan comparando 2 grupos (el grupo que tuvo la modificacion en la UI del checkout y el que no )y al ser una variables binaria es decir 1 o 0 (conviritio o no convirtio)
**Nivel de significancia alpha:** 0.05

In [166]:
#Primero se debe importar la libreria (from statsmodels.stats.proportion import proportions_ztest) esta se importo en las primeras celdas y se los datos
ex_checkout = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
ex_checkout.head(7)

#contar numero de exitos y numero total de registros
conversiones = ex_checkout.groupby('variante')['convirtio'].sum()
totales = ex_checkout.groupby('variante')['convirtio'].count()
totales


,convirtio
variante,
control,4965
tratamiento,5035


In [167]:
#Pasar los valores a fomrato lista
exitos = [conversiones['tratamiento'], conversiones['control']]
observaciones = [totales['tratamiento'], totales['control']]

In [168]:
#Aplicar la prueba y visualizar resultados
z_stat, p_value = proportions_ztest(exitos, observaciones)
print(f"Estadistico Z:{z_stat}")
print(f"Valor P: {p_value}")

#Interpretar resultados
alpha =0.05
if p_value<alpha:
    print('Se rechaza la hipotesis nula: La modificación tuvo impacto en la tasa de conversión')
else:
    print('No se rechaza la hipotesis nula: No hay evidencia suficiente de que la modificación tuvo impacto en la tasa de conversión')

Estadistico Z:0.8132782986429474
Valor P: 0.41605851639119995
No se rechaza la hipotesis nula: No hay evidencia suficiente de que la modificación tuvo impacto en la tasa de conversión


In [169]:
#Se obtiene tasas de éxito
tasa_tratamiento  = exitos[0]/observaciones[0]
tasa_control =  exitos[1]/observaciones[1]
print(f"Tasa de conversión tratamiento:{tasa_tratamiento:.2%}")
print(f"Tasa de conversión control:{tasa_control:.2%}")

#Interpretación resultados
if tasa_tratamiento > tasa_control:
    print(f"\nLa modificación de la UI del checkout tiene una mayor tasa de conversión({tasa_tratamiento-tasa_control:.2%}).")
elif tasa_control > tasa_tratamiento:
    print(f"\nEl diseño actual de la interfaz del checkout tiene una mayor tasa de conversión({tasa_control-tasa_tratamiento:.2%}).")
else:
    print("\nAmbos diseños de interfaz tienen la misma tasa de conversión")

Tasa de conversión tratamiento:16.29%
Tasa de conversión control:15.69%

La modificación de la UI del checkout tiene una mayor tasa de conversión(0.60%).


La tasa de conversión fue de 16,29 % en el grupo de tratamiento y de 15,69 % en el grupo de control, una diferencia de 0,60 puntos porcentuales. Sin embargo, el valor p fue aproximadamente 0,416, mayor que 0,05, por lo que no rechazo la hipótesis nula.

Con estos datos no encontré evidencia suficiente para afirmar que el cambio del checkout haya modificado la conversión. Esto no demuestra que ambas versiones sean iguales. Por ahora, recomendaría mantener la versión anterior mientras se evalúan nuevos cambios.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
https://public.tableau.com/views/ProyectoFinalbootcampDAsprint12/Dashboard1?:language=es-ES&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link